# Computer Vision - Assignment 2
## Benchmarking Foundation Models for Few-Shot Classification

---

## Table of Contents

1. Introduction  
2. Background  
3. Experimental Methodology  
4. Colab Setup  
5. Global Configuration  
6. Dataset Loading  
7. Backbone Loading  
8. Feature Extraction and Caching  
9. Episode Generation  
10. Few-Shot Classifiers  
11. Optimized Benchmark  
12. Results and Tables  
13. Visualizations  
14. Training Dynamics  
15. Feature-Space Visualization  
16. Optional Task 5  
17. Discussion  
18. Limitations  
19. Conclusion  
20. References

# 1. Introduction

This notebook studies few-shot image classification with frozen visual foundation models.

The main question is simple:

> If we already have a strong pre-trained vision model, how well can it classify new classes when it receives only a few labeled examples per class?

We evaluate three frozen backbones:

- **DINOv2-small**
- **CLIP-RN50**
- **ConvNeXt-Tiny**

We evaluate them on two datasets:

- **Stanford Cars**
- **EuroSAT**

For each backbone and dataset, we extract image embeddings once. Then we train small classification heads on few-shot support sets and test them on query sets.

The notebook is designed for Google Colab. It uses caching, checkpointing, and a runtime-aware benchmark structure so expensive work is not repeated.

# 2. Background

## 2.1 Few-Shot Classification

Few-shot classification means that the classifier receives only a small number of labeled examples for each class.

In this assignment, the number of support examples per class is:

$$K \in \{1, 2, 4, 16\}$$

A single test task is called an **episode**. Each episode contains a **support set**, used to build the temporary classifier, and a **query set**, used only for evaluation.

$$S = \{(x_i, y_i)\}_{i=1}^{C \cdot K}$$

$$Q = \{(x_j, y_j)\}_{j=1}^{C \cdot 15}$$

The classifier must learn only from the support set. Query labels are used only to measure accuracy.

## 2.2 Frozen Backbones and Feature Embeddings

A backbone maps an image into a feature vector.

$$z = f(x)$$

where $x$ is an input image, $f$ is the frozen backbone, and $z$ is the feature embedding.

The backbone is not trained in this assignment. To save time, every image is passed through each backbone only once. The resulting features are saved to disk and reused later.

## 2.3 Classification Heads

### Prototypical Classifier

Intuition: each class is represented by the mean of its support embeddings.

$$p_c = \frac{1}{K}\sum_{x_i \in S_c} f(x_i)$$

A query is assigned to the closest prototype. In this notebook, cosine similarity is used, so features are normalized once before repeated use.

### Ridge Regression

Intuition: fit a regularized linear classifier on the support embeddings.

$$W = (X^T X + \lambda I)^{-1}X^T Y$$

This can be slow when the feature dimension is large. The notebook uses an adaptive Ridge implementation. It uses the dual form when the support set is smaller than the feature dimension, and the primal form otherwise.

### Learned Linear Classifier

Intuition: train a small linear layer on top of frozen features.

$$z_i = W^T f(x_i) + b$$

The model is trained with cross-entropy loss on the support set only. The implementation trains directly on tensors and uses early stopping.

# 3. Experimental Methodology

The full benchmark contains:

$$2 \text{ datasets} \times 3 \text{ backbones} \times 3 \text{ heads} \times 4 \text{ K values}$$

This gives 72 main configurations.

The target full benchmark uses 10,000 episodes per configuration. The notebook also supports smaller modes for debugging:

- **debug:** small number of episodes;
- **medium:** medium number of episodes;
- **full:** full benchmark.

The exact number of episodes is stored in the result table. No result is reported without its episode count.

## Runtime Design Decision

The benchmark is organized around a dataset/backbone pair. For one such pair, the cached feature tensor is loaded once and reused for all K-shot values. This avoids loading the same feature file four times for K = 1, 2, 4, and 16. The result files are still saved separately for each K value, so the benchmark can resume from partial progress without repeating completed work.

# 4. Colab Setup

This cell installs the required packages.

It exists because a fresh Google Colab session may not contain all libraries used in the assignment.

Expected output: package installation logs.

In [ ]:
!pip -q install datasets transformers accelerate timm scikit-learn umap-learn git+https://github.com/openai/CLIP.git

This cell imports all Python libraries used in the notebook.

Expected output: a short confirmation message.

In [ ]:
import os
import sys
import json
import time
import math
import random
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

from datasets import load_dataset, concatenate_datasets
from transformers import AutoImageProcessor, AutoModel

from sklearn.manifold import TSNE

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    UMAP_AVAILABLE = False

try:
    import clip
    CLIP_AVAILABLE = True
except Exception:
    CLIP_AVAILABLE = False

warnings.filterwarnings("ignore")

print("Imports completed.")
print("CUDA available:", torch.cuda.is_available())

This cell mounts Google Drive.

It exists because cached features and benchmark results should survive Colab runtime resets.

Expected output: Google Drive mount messages.

In [ ]:
from google.colab import drive

MOUNT_GOOGLE_DRIVE = True

if MOUNT_GOOGLE_DRIVE:
    drive.mount("/content/drive")
else:
    print("Google Drive mounting is disabled.")

# 5. Global Configuration

This cell defines all important runtime settings in one place.

It exists so that the notebook can switch between debug, medium, and full runs without editing many cells.

Expected output: the selected mode, device, and directory paths.

In [ ]:
# Main runtime flags
RUN_FEATURE_EXTRACTION = False # True if first time, otherwise False
RUN_EPISODE_GENERATION = True
RUN_BENCHMARK = True
RUN_TASK5_BENCHMARK = True
RUN_VISUALIZATIONS = True

# Use: "debug", "medium", or "full"
RUN_MODE = "full"

EPISODES_BY_MODE = {
    "debug": 100,
    "medium": 1000,
    "full": 10000,
}

NUM_EPISODES = EPISODES_BY_MODE[RUN_MODE]

SEED = 42
K_SHOTS = [1, 2, 4, 16]
QUERY_PER_CLASS = 15

DATASET_NAMES = ["stanford_cars", "eurosat"]
BACKBONE_NAMES = ["dinov2_small", "clip_rn50", "convnext_tiny"]
HEAD_NAMES = ["prototype", "ridge", "linear"]

LINEAR_MAX_EPOCHS = 10
LINEAR_LR = 0.05
LINEAR_WEIGHT_DECAY = 1e-4
LINEAR_PATIENCE = 3
LINEAR_MIN_DELTA = 1e-4

RIDGE_LAMBDA = 1.0

PROJECT_DIR = Path("/content/drive/MyDrive/cv_assignment2_optimized")
#CACHE_DIR = PROJECT_DIR / "cached_features"
CACHE_DIR = Path("/content/drive/MyDrive/cv_assignment2/cached_features")  # old valid features
EPISODE_DIR = PROJECT_DIR / "episodes"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURE_DIR = PROJECT_DIR / "figures"
CURVE_DIR = PROJECT_DIR / "training_curves"

for directory in [PROJECT_DIR, CACHE_DIR, EPISODE_DIR, RESULTS_DIR, FIGURE_DIR, CURVE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Run mode:", RUN_MODE)
print("Number of episodes:", NUM_EPISODES)
print("Device:", DEVICE)
print("Project directory:", PROJECT_DIR)

This cell fixes random seeds.

It exists so sampled episodes and model training are more reproducible.

Expected output: a confirmation message.

In [ ]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_global_seed(SEED)
print("Seed fixed:", SEED)

# 6. Helper Functions for Paths and Logging

This cell defines small helper functions for file paths and timing.

It exists to avoid duplicated path logic in later sections.

Expected output: no output unless the functions are called later.

In [ ]:
def feature_cache_path(dataset_name: str, backbone_name: str) -> Path:
    return CACHE_DIR / f"features__{dataset_name}__{backbone_name}.pt"

def episode_cache_path(dataset_name: str, k_shot: int, num_episodes: int) -> Path:
    return EPISODE_DIR / f"episodes__{dataset_name}__K{k_shot}__N{num_episodes}__seed{SEED}.pt"

def result_path(dataset_name: str, backbone_name: str, k_shot: int, num_episodes: int) -> Path:
    return RESULTS_DIR / f"results__{dataset_name}__{backbone_name}__K{k_shot}__N{num_episodes}.csv"

def task5_result_path(dataset_name: str, backbone_name: str, k_shot: int, num_episodes: int) -> Path:
    return RESULTS_DIR / f"task5__{dataset_name}__{backbone_name}__K{k_shot}__N{num_episodes}.csv"

def timing_path() -> Path:
    return RESULTS_DIR / f"timing__{RUN_MODE}__N{NUM_EPISODES}.csv"

def now_seconds() -> float:
    return time.perf_counter()

def elapsed_seconds(start_time: float) -> float:
    return time.perf_counter() - start_time

# 7. Dataset Loading

This cell defines dataset-loading helpers.

It exists because different Hugging Face datasets may use different column names for images and labels.

Expected output: no output yet.

In [ ]:
def find_first_existing_column(dataset, candidates):
    for name in candidates:
        if name in dataset.column_names:
            return name
    raise ValueError(
        f"None of the candidate columns exist: {candidates}. "
        f"Available columns: {dataset.column_names}"
    )

def standardize_dataset_columns(dataset, image_column: str, label_column: str):
    needed_columns = [image_column, label_column]
    dataset = dataset.select_columns(needed_columns)

    if image_column != "image":
        dataset = dataset.rename_column(image_column, "image")

    if label_column != "label":
        dataset = dataset.rename_column(label_column, "label")

    return dataset

def get_class_names_from_dataset(dataset) -> Optional[List[str]]:
    label_feature = dataset.features.get("label", None)
    if hasattr(label_feature, "names"):
        return list(label_feature.names)
    return None

def load_dataset_with_fallback(candidates: List[Tuple[str, Optional[str]]]):
    last_error = None
    for dataset_id, config_name in candidates:
        try:
            if config_name is None:
                return load_dataset(dataset_id)
            return load_dataset(dataset_id, config_name)
        except Exception as error:
            last_error = error
            print(f"Could not load {dataset_id} {config_name}: {error}")
    raise RuntimeError(f"All dataset loading attempts failed. Last error: {last_error}")


This cell loads Stanford Cars and EuroSAT.

It concatenates available train/test/validation splits into one evaluation pool, as required by the assignment.

Expected output: dataset sizes and number of classes.

In [ ]:
def load_stanford_cars_dataset():
    candidates = [
        ("tanganke/stanford_cars", None),
        ("HuggingFaceM4/Stanford-Cars", None),
    ]
    raw = load_dataset_with_fallback(candidates)
    splits = [raw[key] for key in raw.keys()]
    dataset = concatenate_datasets(splits) if len(splits) > 1 else splits[0]
    # Expanded candidate lists for image and label columns
    image_col = find_first_existing_column(dataset, ["image", "img", "pixel_values", "pixels"])
    label_col = find_first_existing_column(dataset, ["label", "labels", "class", "fine_label", "target"])
    dataset = standardize_dataset_columns(dataset, image_col, label_col)
    class_names = get_class_names_from_dataset(dataset)
    return dataset, class_names

def load_eurosat_dataset():
    candidates = [
        ("blanchon/EuroSAT_RGB", None),
        ("timm/eurosat-rgb", None),
        ("eurosat", "rgb"),
    ]
    raw = load_dataset_with_fallback(candidates)
    splits = [raw[key] for key in raw.keys()]
    dataset = concatenate_datasets(splits) if len(splits) > 1 else splits[0]
    # Expanded candidate lists for image and label columns
    image_col = find_first_existing_column(dataset, ["image", "img", "pixel_values", "pixels"])
    label_col = find_first_existing_column(dataset, ["label", "labels", "class", "fine_label", "target"])
    dataset = standardize_dataset_columns(dataset, image_col, label_col)
    class_names = get_class_names_from_dataset(dataset)
    return dataset, class_names

def load_all_datasets() -> Dict[str, Dict[str, Any]]:
    datasets_dict = {}
    cars_dataset, cars_classes = load_stanford_cars_dataset()
    eurosat_dataset, eurosat_classes = load_eurosat_dataset()
    datasets_dict["stanford_cars"] = {"dataset": cars_dataset, "class_names": cars_classes}
    datasets_dict["eurosat"] = {"dataset": eurosat_dataset, "class_names": eurosat_classes}
    return datasets_dict

all_datasets = load_all_datasets()

for name, pack in all_datasets.items():
    dataset = pack["dataset"]
    labels = np.array(dataset["label"])
    print(name)
    print("  samples:", len(dataset))
    print("  classes:", len(np.unique(labels)))

# 8. Backbone Loading

This cell defines functions that load the three frozen visual backbones.

It exists so feature extraction can use one unified interface.

Expected output: no output yet.

In [ ]:
@dataclass
class BackbonePack:
    name: str
    model: Any
    processor: Any
    kind: str

def load_dinov2_small(device: torch.device) -> BackbonePack:
    processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
    model = AutoModel.from_pretrained("facebook/dinov2-small").to(device)
    model.eval()
    return BackbonePack("dinov2_small", model, processor, "transformers_cls")

def load_convnext_tiny(device: torch.device) -> BackbonePack:
    processor = AutoImageProcessor.from_pretrained("facebook/convnext-tiny-224")
    model = AutoModel.from_pretrained("facebook/convnext-tiny-224").to(device)
    model.eval()
    return BackbonePack("convnext_tiny", model, processor, "transformers_pool")

def load_clip_rn50(device: torch.device) -> BackbonePack:
    if not CLIP_AVAILABLE:
        raise RuntimeError("OpenAI CLIP is not available. Check the installation cell.")
    model, preprocess = clip.load("RN50", device=device)
    model.eval()
    return BackbonePack("clip_rn50", model, preprocess, "openai_clip")

def load_backbone(backbone_name: str, device: torch.device) -> BackbonePack:
    if backbone_name == "dinov2_small":
        return load_dinov2_small(device)
    if backbone_name == "convnext_tiny":
        return load_convnext_tiny(device)
    if backbone_name == "clip_rn50":
        return load_clip_rn50(device)
    raise ValueError(f"Unknown backbone: {backbone_name}")

This cell defines image preprocessing and feature extraction for one batch.

It exists because each backbone expects images in a different format.

Expected output: no output yet.

In [ ]:
def ensure_rgb_image(image) -> Image.Image:
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    return Image.fromarray(np.array(image)).convert("RGB")

def extract_transformers_features(images: List[Image.Image], backbone: BackbonePack, device: torch.device) -> torch.Tensor:
    inputs = backbone.processor(images=images, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        outputs = backbone.model(**inputs)
    if backbone.kind == "transformers_cls":
        features = outputs.last_hidden_state[:, 0]
    else:
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            features = outputs.pooler_output
        else:
            features = outputs.last_hidden_state.mean(dim=1)
    return features.detach().float().cpu()

def extract_clip_features(images: List[Image.Image], backbone: BackbonePack, device: torch.device) -> torch.Tensor:
    processed = [backbone.processor(image) for image in images]
    image_tensor = torch.stack(processed).to(device)
    with torch.no_grad():
        features = backbone.model.encode_image(image_tensor)
    return features.detach().float().cpu()

def extract_batch_features(images: List[Image.Image], backbone: BackbonePack, device: torch.device) -> torch.Tensor:
    if backbone.kind == "openai_clip":
        return extract_clip_features(images, backbone, device)
    return extract_transformers_features(images, backbone, device)

# 9. Feature Extraction and Caching

This cell extracts features for one dataset and one backbone.

It skips extraction if the cache file already exists.

Expected output: progress bars and a saved `.pt` file for each dataset/backbone pair.

In [ ]:
def extract_features_for_dataset(dataset_name: str, dataset_pack: Dict[str, Any], backbone_name: str, batch_size: int, device: torch.device) -> Path:
    output_path = feature_cache_path(dataset_name, backbone_name)
    if output_path.exists():
        print(f"Cache already exists, skipping: {output_path}")
        return output_path

    dataset = dataset_pack["dataset"]
    class_names = dataset_pack["class_names"]
    backbone = load_backbone(backbone_name, device)

    all_features = []
    all_labels = []
    start = now_seconds()

    for start_idx in tqdm(range(0, len(dataset), batch_size), desc=f"{dataset_name} / {backbone_name}"):
        end_idx = min(start_idx + batch_size, len(dataset))
        batch = dataset.select(range(start_idx, end_idx))
        images = [ensure_rgb_image(image) for image in batch["image"]]
        labels = torch.tensor(batch["label"], dtype=torch.long)
        features = extract_batch_features(images, backbone, device)
        all_features.append(features)
        all_labels.append(labels.cpu())

    features_tensor = torch.cat(all_features, dim=0).float()
    labels_tensor = torch.cat(all_labels, dim=0).long()

    payload = {
        "dataset_name": dataset_name,
        "backbone_name": backbone_name,
        "features": features_tensor,
        "features_normalized": F.normalize(features_tensor, dim=1),
        "labels": labels_tensor,
        "class_names": class_names,
        "num_samples": int(features_tensor.shape[0]),
        "feature_dim": int(features_tensor.shape[1]),
        "extraction_seconds": elapsed_seconds(start),
    }
    torch.save(payload, output_path)
    print(f"Saved: {output_path}")
    print("Shape:", tuple(features_tensor.shape))
    print("Time seconds:", payload["extraction_seconds"])

    del backbone
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return output_path

This cell runs feature extraction for all dataset/backbone pairs if `RUN_FEATURE_EXTRACTION=True`.

It exists because feature extraction is expensive and should only be done once.

Expected output: six cached feature files, or skip messages if they already exist.

In [ ]:
FEATURE_EXTRACTION_BATCH_SIZE = 64

if RUN_FEATURE_EXTRACTION:
    for dataset_name in DATASET_NAMES:
        for backbone_name in BACKBONE_NAMES:
            extract_features_for_dataset(
                dataset_name=dataset_name,
                dataset_pack=all_datasets[dataset_name],
                backbone_name=backbone_name,
                batch_size=FEATURE_EXTRACTION_BATCH_SIZE,
                device=DEVICE,
            )
else:
    print("Feature extraction is disabled.")
    print("Set RUN_FEATURE_EXTRACTION = True if cached feature files do not exist.")

This cell checks that all required feature files exist.

It exists because benchmarking cannot start without cached embeddings.

Expected output: either a success message or a clear stop message.

In [ ]:
def check_required_feature_files() -> List[Path]:
    missing = []
    for dataset_name in DATASET_NAMES:
        for backbone_name in BACKBONE_NAMES:
            path = feature_cache_path(dataset_name, backbone_name)
            if not path.exists():
                missing.append(path)
    return missing

missing_feature_files = check_required_feature_files()

if missing_feature_files:
    print("Execution must stop here.")
    print("Reason: the benchmark requires cached feature files, but some files are missing.")
    print("What you must do:")
    print("1. Set RUN_FEATURE_EXTRACTION = True in the configuration cell.")
    print("2. Run the feature extraction cells until they finish.")
    print("3. Confirm that the following files are created:")
    for path in missing_feature_files:
        print("   -", path)
    print("4. After the files exist, rerun this cell and continue.")
    raise SystemExit("Missing cached feature files. Run feature extraction before benchmarking.")

print("All required feature files exist.")

# 10. Loading Cached Features

This cell defines a loader for cached feature tensors.

It moves tensors to the selected device once and keeps both raw and normalized features.

Expected output: no output yet.

In [ ]:
def load_feature_cache(dataset_name: str, backbone_name: str, device: torch.device) -> Dict[str, Any]:
    path = feature_cache_path(dataset_name, backbone_name)
    payload = torch.load(path, map_location="cpu")
    features = payload["features"].float().to(device)
    labels = payload["labels"].long().to(device)
    if "features_normalized" in payload:
        features_normalized = payload["features_normalized"].float().to(device)
    else:
        features_normalized = F.normalize(features, dim=1)
    return {
        "dataset_name": dataset_name,
        "backbone_name": backbone_name,
        "features": features,
        "features_normalized": features_normalized,
        "labels": labels,
        "class_names": payload.get("class_names", None),
        "feature_dim": int(features.shape[1]),
        "num_samples": int(features.shape[0]),
    }

def build_class_to_indices(labels_cpu: torch.Tensor) -> Dict[int, torch.Tensor]:
    class_to_indices = {}
    unique_labels = torch.unique(labels_cpu).tolist()
    for class_id in unique_labels:
        indices = torch.where(labels_cpu == int(class_id))[0]
        class_to_indices[int(class_id)] = indices.cpu()
    return class_to_indices

# 11. Episode Generation

This cell defines episode generation.

It exists to sample the few-shot tasks once and reuse the same episodes for all heads. This makes the comparison fair and avoids repeated sampling overhead.

Expected output: no output yet.

In [ ]:
def validate_class_counts(class_to_indices: Dict[int, torch.Tensor], k_shot: int, query_per_class: int) -> List[int]:
    valid_classes = []
    required = k_shot + query_per_class
    for class_id, indices in class_to_indices.items():
        if len(indices) >= required:
            valid_classes.append(class_id)
    if len(valid_classes) != len(class_to_indices):
        print("Warning: some classes do not have enough samples.")
        print("Using valid classes:", len(valid_classes), "out of", len(class_to_indices))
    return sorted(valid_classes)

def generate_single_episode(class_to_indices: Dict[int, torch.Tensor], class_ids: List[int], k_shot: int, query_per_class: int, rng: np.random.Generator) -> Dict[str, torch.Tensor]:
    support_indices = []
    query_indices = []
    support_labels = []
    query_labels = []
    for local_label, class_id in enumerate(class_ids):
        available = class_to_indices[class_id].numpy()
        selected = rng.choice(available, size=k_shot + query_per_class, replace=False)
        support_selected = selected[:k_shot]
        query_selected = selected[k_shot:]
        support_indices.extend(support_selected.tolist())
        query_indices.extend(query_selected.tolist())
        support_labels.extend([local_label] * k_shot)
        query_labels.extend([local_label] * query_per_class)
    return {
        "support_indices": torch.tensor(support_indices, dtype=torch.long),
        "query_indices": torch.tensor(query_indices, dtype=torch.long),
        "support_labels": torch.tensor(support_labels, dtype=torch.long),
        "query_labels": torch.tensor(query_labels, dtype=torch.long),
        "class_ids": torch.tensor(class_ids, dtype=torch.long),
    }

def generate_episodes_for_dataset(dataset_name: str, labels: torch.Tensor, k_shot: int, num_episodes: int, query_per_class: int, seed: int) -> Path:
    output_path = episode_cache_path(dataset_name, k_shot, num_episodes)
    if output_path.exists():
        print(f"Episode cache already exists, skipping: {output_path}")
        return output_path
    labels_cpu = labels.detach().cpu()
    class_to_indices = build_class_to_indices(labels_cpu)
    valid_classes = validate_class_counts(class_to_indices, k_shot, query_per_class)
    rng = np.random.default_rng(seed + 1000 * k_shot)
    episodes = []
    for _ in tqdm(range(num_episodes), desc=f"Generating episodes {dataset_name} K={k_shot}"):
        episode = generate_single_episode(class_to_indices, valid_classes, k_shot, query_per_class, rng)
        episodes.append(episode)
    payload = {
        "dataset_name": dataset_name,
        "k_shot": k_shot,
        "num_episodes": num_episodes,
        "query_per_class": query_per_class,
        "seed": seed,
        "num_classes": len(valid_classes),
        "episodes": episodes,
    }
    torch.save(payload, output_path)
    print("Saved episodes:", output_path)
    return output_path

This cell pre-generates episodes for each dataset and each K value.

It uses labels from the first available backbone cache for each dataset.

Expected output: episode cache files, or skip messages if they already exist.

In [ ]:
if RUN_EPISODE_GENERATION:
    for dataset_name in DATASET_NAMES:
        reference_payload = torch.load(feature_cache_path(dataset_name, BACKBONE_NAMES[0]), map_location="cpu")
        reference_labels = reference_payload["labels"].long()
        for k_shot in K_SHOTS:
            generate_episodes_for_dataset(
                dataset_name=dataset_name,
                labels=reference_labels,
                k_shot=k_shot,
                num_episodes=NUM_EPISODES,
                query_per_class=QUERY_PER_CLASS,
                seed=SEED,
            )
else:
    print("Episode generation is disabled.")

This cell checks that all required episode files exist.

It exists because the optimized benchmark expects fixed pre-generated episodes.

Expected output: either a success message or a clear stop message.

In [ ]:
def check_required_episode_files() -> List[Path]:
    missing = []
    for dataset_name in DATASET_NAMES:
        for k_shot in K_SHOTS:
            path = episode_cache_path(dataset_name, k_shot, NUM_EPISODES)
            if not path.exists():
                missing.append(path)
    return missing

missing_episode_files = check_required_episode_files()

if missing_episode_files:
    print("Execution must stop here.")
    print("Reason: benchmark episodes are missing.")
    print("What you must do:")
    print("1. Set RUN_EPISODE_GENERATION = True.")
    print("2. Run the episode generation cell.")
    print("3. Confirm that these files exist:")
    for path in missing_episode_files:
        print("   -", path)
    raise SystemExit("Missing episode files. Generate episodes before benchmarking.")

print("All required episode files exist.")

# 12. Few-Shot Classifiers

This cell implements accuracy computation and tensor selection for episodes.

It exists because all heads need the same support/query tensors.

Expected output: no output yet.

In [ ]:
def get_episode_tensors(feature_pack: Dict[str, Any], episode: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    support_indices = episode["support_indices"].to(device)
    query_indices = episode["query_indices"].to(device)
    support_y = episode["support_labels"].to(device)
    query_y = episode["query_labels"].to(device)
    return {
        "support_x": feature_pack["features"][support_indices],
        "query_x": feature_pack["features"][query_indices],
        "support_x_norm": feature_pack["features_normalized"][support_indices],
        "query_x_norm": feature_pack["features_normalized"][query_indices],
        "support_y": support_y,
        "query_y": query_y,
        "num_classes": int(torch.max(support_y).item()) + 1,
    }

def compute_accuracy(predictions: torch.Tensor, labels: torch.Tensor) -> float:
    return float((predictions == labels).float().mean().item())

This cell implements the Prototypical classifier.

Intuition: compute one mean feature vector per class and classify each query by the closest prototype.

Expected output: no output yet.

In [ ]:
def compute_class_prototypes(
    support_x_norm: torch.Tensor,
    support_y: torch.Tensor,
    num_classes: int,
) -> torch.Tensor:
    one_hot = F.one_hot(support_y, num_classes=num_classes).float()
    class_counts = one_hot.sum(dim=0).clamp_min(1.0).unsqueeze(1)

    prototypes = one_hot.T @ support_x_norm
    prototypes = prototypes / class_counts

    prototypes = F.normalize(prototypes, dim=1)
    return prototypes

@torch.no_grad()
def predict_prototype(support_x_norm: torch.Tensor, support_y: torch.Tensor, query_x_norm: torch.Tensor, num_classes: int) -> torch.Tensor:
    prototypes = compute_class_prototypes(support_x_norm, support_y, num_classes)
    scores = query_x_norm @ prototypes.T
    return torch.argmax(scores, dim=1)

This cell implements Ridge Regression.

It uses the dual formulation when the support set is smaller than the feature dimension. Otherwise, it uses the primal formulation.

Expected output: no output yet.

In [ ]:
def one_hot(labels: torch.Tensor, num_classes: int) -> torch.Tensor:
    return F.one_hot(labels, num_classes=num_classes).float()

@torch.no_grad()
def predict_ridge(support_x: torch.Tensor, support_y: torch.Tensor, query_x: torch.Tensor, num_classes: int, ridge_lambda: float) -> torch.Tensor:
    x_support = support_x.float()
    x_query = query_x.float()
    y_support = one_hot(support_y, num_classes)
    n_support, feature_dim = x_support.shape
    if n_support <= feature_dim:
        kernel = x_support @ x_support.T
        identity = torch.eye(n_support, device=x_support.device, dtype=x_support.dtype)
        alpha = torch.linalg.solve(kernel + ridge_lambda * identity, y_support)
        scores = x_query @ x_support.T @ alpha
    else:
        gram = x_support.T @ x_support
        identity = torch.eye(feature_dim, device=x_support.device, dtype=x_support.dtype)
        weights = torch.linalg.solve(gram + ridge_lambda * identity, x_support.T @ y_support)
        scores = x_query @ weights
    return torch.argmax(scores, dim=1)

This cell implements the learned Linear classifier.

It trains only a single linear layer on the support set and uses early stopping to avoid unnecessary epochs.

Expected output: no output yet.

In [ ]:
def initialize_linear_parameters(feature_dim: int, num_classes: int, device: torch.device):
    return torch.nn.Linear(feature_dim, num_classes).to(device)

def train_linear_head(
    support_x: torch.Tensor,
    support_y: torch.Tensor,
    query_x: torch.Tensor,
    query_y: torch.Tensor,
    num_classes: int,
    max_epochs: int,
    learning_rate: float,
    weight_decay: float,
    patience: int,
    min_delta: float,
    return_curve: bool = False,
):
    feature_dim = support_x.shape[1]
    model = initialize_linear_parameters(feature_dim, num_classes, support_x.device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    best_loss = float("inf")
    epochs_without_improvement = 0
    curve_rows = []

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(support_x)
        loss = F.cross_entropy(logits, support_y)
        loss.backward()
        optimizer.step()

        loss_value = float(loss.item())

        # Compute training curves only when explicitly requested.
        if return_curve:
            model.eval()
            with torch.no_grad():
                train_pred = torch.argmax(model(support_x), dim=1)
                query_pred = torch.argmax(model(query_x), dim=1)

                train_acc = compute_accuracy(train_pred, support_y)
                query_acc = compute_accuracy(query_pred, query_y)

            curve_rows.append(
                {
                    "epoch": epoch + 1,
                    "loss": loss_value,
                    "support_accuracy": train_acc,
                    "query_accuracy": query_acc,
                }
            )

        if best_loss - loss_value > min_delta:
            best_loss = loss_value
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.eval()
    with torch.no_grad():
        predictions = torch.argmax(model(query_x), dim=1)

    curve_df = pd.DataFrame(curve_rows) if return_curve else None

    return predictions, curve_df

# 13. Optimized Benchmark Runner

This section defines the optimized benchmark logic.

The benchmark unit is a dataset/backbone pair, not a dataset/backbone/K block. This matters because the same cached feature file is needed for all K values. The code loads the feature cache once for a dataset/backbone pair, keeps it on the selected device, and then evaluates all K values inside the same loaded context.

Main optimizations:

1. Features are loaded once for each dataset/backbone pair.
2. Episodes are pre-generated and cached.
3. Each episode is sampled once.
4. All heads run on the same episode before moving to the next episode.
5. Ridge uses an efficient dual formulation when this is cheaper.
6. The linear classifier uses early stopping.
7. Results are checkpointed after each dataset/backbone/K result file.
8. Existing completed K results are skipped, so the benchmark can resume.

Expected output: no output yet. The next code cell only defines functions.

In [ ]:
def load_episodes(dataset_name: str, k_shot: int, num_episodes: int) -> List[Dict[str, torch.Tensor]]:
    payload = torch.load(episode_cache_path(dataset_name, k_shot, num_episodes), map_location="cpu")
    return payload["episodes"]

def should_record_linear_curve(dataset_name: str, backbone_name: str, k_shot: int, episode_idx: int) -> bool:
    return episode_idx == 0 and k_shot in [1, 16]

def save_linear_curve_if_needed(curve_df: Optional[pd.DataFrame], dataset_name: str, backbone_name: str, k_shot: int) -> None:
    if curve_df is None or curve_df.empty:
        return
    output_path = CURVE_DIR / f"linear_curve__{dataset_name}__{backbone_name}__K{k_shot}.csv"
    curve_df.to_csv(output_path, index=False)

def run_heads_on_episode(tensors: Dict[str, torch.Tensor], dataset_name: str, backbone_name: str, k_shot: int, episode_idx: int):
    accuracies = {}
    runtimes = {}

    start = now_seconds()
    pred_proto = predict_prototype(
        tensors["support_x_norm"],
        tensors["support_y"],
        tensors["query_x_norm"],
        tensors["num_classes"],
    )
    accuracies["prototype"] = compute_accuracy(pred_proto, tensors["query_y"])
    runtimes["prototype"] = elapsed_seconds(start)

    start = now_seconds()
    pred_ridge = predict_ridge(
        tensors["support_x"],
        tensors["support_y"],
        tensors["query_x"],
        tensors["num_classes"],
        RIDGE_LAMBDA,
    )
    accuracies["ridge"] = compute_accuracy(pred_ridge, tensors["query_y"])
    runtimes["ridge"] = elapsed_seconds(start)

    start = now_seconds()
    return_curve = should_record_linear_curve(dataset_name, backbone_name, k_shot, episode_idx)
    pred_linear, curve_df = train_linear_head(
        support_x=tensors["support_x"],
        support_y=tensors["support_y"],
        query_x=tensors["query_x"],
        query_y=tensors["query_y"],
        num_classes=tensors["num_classes"],
        max_epochs=LINEAR_MAX_EPOCHS,
        learning_rate=LINEAR_LR,
        weight_decay=LINEAR_WEIGHT_DECAY,
        patience=LINEAR_PATIENCE,
        min_delta=LINEAR_MIN_DELTA,
        return_curve=return_curve,
    )
    accuracies["linear"] = compute_accuracy(pred_linear, tensors["query_y"])
    runtimes["linear"] = elapsed_seconds(start)

    if return_curve:
        save_linear_curve_if_needed(curve_df, dataset_name, backbone_name, k_shot)

    return accuracies, runtimes

def summarize_block_results(
    dataset_name: str,
    backbone_name: str,
    k_shot: int,
    num_episodes: int,
    head_to_scores: Dict[str, List[float]],
    head_to_times: Dict[str, List[float]],
    total_seconds: float,
) -> pd.DataFrame:
    rows = []
    for head_name in HEAD_NAMES:
        scores = np.array(head_to_scores[head_name], dtype=np.float32)
        times = np.array(head_to_times[head_name], dtype=np.float32)
        rows.append({
            "dataset": dataset_name,
            "backbone": backbone_name,
            "head": head_name,
            "k_shot": k_shot,
            "num_episodes": num_episodes,
            "mean_accuracy": float(scores.mean()),
            "std_accuracy": float(scores.std(ddof=1)) if len(scores) > 1 else 0.0,
            "mean_head_seconds_per_episode": float(times.mean()),
            "total_block_seconds": float(total_seconds),
            "run_mode": RUN_MODE,
            "seed": SEED,
        })
    return pd.DataFrame(rows)

def run_benchmark_k_from_loaded_features(
    dataset_name: str,
    backbone_name: str,
    k_shot: int,
    num_episodes: int,
    feature_pack: Dict[str, torch.Tensor],
) -> pd.DataFrame:
    output_path = result_path(dataset_name, backbone_name, k_shot, num_episodes)
    if output_path.exists():
        print("Result already exists, skipping:", output_path)
        return pd.read_csv(output_path)

    episodes = load_episodes(dataset_name, k_shot, num_episodes)
    head_to_scores = {head: [] for head in HEAD_NAMES}
    head_to_times = {head: [] for head in HEAD_NAMES}

    start_block = now_seconds()
    description = f"{dataset_name}/{backbone_name}/K={k_shot}"

    for episode_idx, episode in enumerate(tqdm(episodes, desc=description)):
        tensors = get_episode_tensors(feature_pack, episode, DEVICE)
        accuracies, runtimes = run_heads_on_episode(
            tensors=tensors,
            dataset_name=dataset_name,
            backbone_name=backbone_name,
            k_shot=k_shot,
            episode_idx=episode_idx,
        )
        for head_name in HEAD_NAMES:
            head_to_scores[head_name].append(accuracies[head_name])
            head_to_times[head_name].append(runtimes[head_name])

    total_seconds = elapsed_seconds(start_block)
    result_df = summarize_block_results(
        dataset_name=dataset_name,
        backbone_name=backbone_name,
        k_shot=k_shot,
        num_episodes=num_episodes,
        head_to_scores=head_to_scores,
        head_to_times=head_to_times,
        total_seconds=total_seconds,
    )
    result_df.to_csv(output_path, index=False)
    print("Saved result:", output_path)
    return result_df

def run_benchmark_dataset_backbone_block(dataset_name: str, backbone_name: str, num_episodes: int) -> pd.DataFrame:
    print(f"Loading feature cache once for {dataset_name}/{backbone_name}")
    feature_pack = load_feature_cache(dataset_name, backbone_name, DEVICE)

    block_frames = []
    start_pair = now_seconds()

    for k_shot in K_SHOTS:
        block_df = run_benchmark_k_from_loaded_features(
            dataset_name=dataset_name,
            backbone_name=backbone_name,
            k_shot=k_shot,
            num_episodes=num_episodes,
            feature_pack=feature_pack,
        )
        block_frames.append(block_df)

    total_pair_seconds = elapsed_seconds(start_pair)
    print(f"Finished {dataset_name}/{backbone_name} in {total_pair_seconds:.2f} seconds")

    del feature_pack
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.concat(block_frames, ignore_index=True)

def run_benchmark_block(dataset_name: str, backbone_name: str, k_shot: int, num_episodes: int) -> pd.DataFrame:
    print("Compatibility wrapper: loading features for a single K block.")
    feature_pack = load_feature_cache(dataset_name, backbone_name, DEVICE)
    result_df = run_benchmark_k_from_loaded_features(
        dataset_name=dataset_name,
        backbone_name=backbone_name,
        k_shot=k_shot,
        num_episodes=num_episodes,
        feature_pack=feature_pack,
    )
    del feature_pack
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result_df


This cell runs the main benchmark if `RUN_BENCHMARK=True`.

The benchmark now loads each dataset/backbone feature cache only once, then loops over all K values while that feature tensor remains on the selected device. It still saves one CSV file per dataset/backbone/K, so resume behavior is preserved.

Expected output: progress bars and saved result CSV files.

In [ ]:
all_result_frames = []

if RUN_BENCHMARK:
    for dataset_name in DATASET_NAMES:
        for backbone_name in BACKBONE_NAMES:
            pair_df = run_benchmark_dataset_backbone_block(
                dataset_name=dataset_name,
                backbone_name=backbone_name,
                num_episodes=NUM_EPISODES,
            )
            all_result_frames.append(pair_df)

    benchmark_results = pd.concat(all_result_frames, ignore_index=True)
    combined_path = RESULTS_DIR / f"benchmark_results__{RUN_MODE}__N{NUM_EPISODES}.csv"
    benchmark_results.to_csv(combined_path, index=False)
    print("Saved combined results:", combined_path)
    display(benchmark_results)
else:
    print("Benchmark is disabled.")


# 14. Loading Benchmark Results

This cell loads all saved benchmark CSV files for the selected run mode.

It exists so visualizations can be generated without rerunning heavy experiments.

Expected output: a result table.

In [ ]:
def load_available_result_files(num_episodes: int) -> pd.DataFrame:
    frames = []
    for path in sorted(RESULTS_DIR.glob(f"results__*__N{num_episodes}.csv")):
        frames.append(pd.read_csv(path))
    if not frames:
        print("Execution must stop here.")
        print("Reason: no benchmark result CSV files were found for the selected episode count.")
        print("What you must do:")
        print("1. Set RUN_BENCHMARK = True.")
        print("2. Run the benchmark cells.")
        print("3. Confirm that result CSV files are saved in:", RESULTS_DIR)
        raise SystemExit("Missing benchmark results.")
    return pd.concat(frames, ignore_index=True)

results_df = load_available_result_files(NUM_EPISODES)
display(results_df)

# 15. Benchmark Tables

This cell creates a clean benchmark table.

It exists because raw CSV files are useful for storage, but the report needs a readable summary.

Expected output: a sorted table with mean accuracy, standard deviation, episode count, and timing.

In [ ]:
def format_results_table(df: pd.DataFrame) -> pd.DataFrame:
    table = df.copy()
    table["mean_accuracy_percent"] = 100.0 * table["mean_accuracy"]
    table["std_accuracy_percent"] = 100.0 * table["std_accuracy"]
    table = table[["dataset", "backbone", "head", "k_shot", "num_episodes", "mean_accuracy_percent", "std_accuracy_percent", "mean_head_seconds_per_episode", "total_block_seconds"]]
    table = table.sort_values(["dataset", "backbone", "k_shot", "head"]).reset_index(drop=True)
    return table

summary_table = format_results_table(results_df)
display(summary_table)

## Interpretation Notes

The table above should be read carefully.

- `mean_accuracy_percent` is the average query accuracy across sampled episodes.
- `std_accuracy_percent` shows how much accuracy changes across episodes.
- `num_episodes` makes the computation budget explicit.
- Timing columns show which heads are expensive.

No numerical conclusion should be written before the table is actually generated.

# 16. Visualization Helpers

This cell defines plotting helpers.

It exists so all figures use consistent filenames and are saved automatically.

Expected output: no output yet.

In [ ]:
def save_current_figure(filename: str) -> Path:
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print("Saved figure:", path)
    return path

def prepare_plot_df(df: pd.DataFrame) -> pd.DataFrame:
    plot_df = df.copy()
    plot_df["accuracy_percent"] = 100.0 * plot_df["mean_accuracy"]
    return plot_df

# 17. Scaling Plots

This cell creates line plots of accuracy versus the number of shots.

It exists because the assignment asks to study how performance changes when K increases.

Expected output: one saved scaling plot for each dataset/backbone pair.

In [ ]:
def plot_scaling_curves(df: pd.DataFrame) -> None:
    plot_df = prepare_plot_df(df)
    for dataset_name in sorted(plot_df["dataset"].unique()):
        for backbone_name in sorted(plot_df["backbone"].unique()):
            subset = plot_df[(plot_df["dataset"] == dataset_name) & (plot_df["backbone"] == backbone_name)]
            if subset.empty:
                continue
            plt.figure(figsize=(7, 5))
            for head_name in HEAD_NAMES:
                head_subset = subset[subset["head"] == head_name].sort_values("k_shot")
                if head_subset.empty:
                    continue
                plt.plot(head_subset["k_shot"], head_subset["accuracy_percent"], marker="o", label=head_name)
            plt.xlabel("Number of shots K")
            plt.ylabel("Mean accuracy (%)")
            plt.title(f"Accuracy Scaling - {dataset_name} / {backbone_name}")
            plt.xticks(K_SHOTS)
            plt.grid(True, alpha=0.3)
            plt.legend()
            filename = f"scaling__{dataset_name}__{backbone_name}__N{NUM_EPISODES}.png"
            save_current_figure(filename)
            plt.show()

if RUN_VISUALIZATIONS:
    plot_scaling_curves(results_df)
else:
    print("Visualizations are disabled.")

# 18. Bar Charts

This cell creates bar charts comparing heads and backbones.

It exists to give a compact view of the main benchmark results.

Expected output: saved bar chart figures.

In [ ]:
def plot_bar_charts(df: pd.DataFrame) -> None:
    plot_df = prepare_plot_df(df)
    for dataset_name in sorted(plot_df["dataset"].unique()):
        subset = plot_df[plot_df["dataset"] == dataset_name]
        for k_shot in sorted(subset["k_shot"].unique()):
            k_subset = subset[subset["k_shot"] == k_shot]
            labels = []
            values = []
            for backbone_name in BACKBONE_NAMES:
                for head_name in HEAD_NAMES:
                    row = k_subset[(k_subset["backbone"] == backbone_name) & (k_subset["head"] == head_name)]
                    if row.empty:
                        continue
                    labels.append(f"{backbone_name}\n{head_name}")
                    values.append(float(row["accuracy_percent"].iloc[0]))
            plt.figure(figsize=(12, 5))
            plt.bar(range(len(values)), values)
            plt.xticks(range(len(values)), labels, rotation=45, ha="right")
            plt.ylabel("Mean accuracy (%)")
            plt.title(f"Benchmark Comparison - {dataset_name}, K={k_shot}")
            plt.grid(True, axis="y", alpha=0.3)
            filename = f"bar__{dataset_name}__K{k_shot}__N{NUM_EPISODES}.png"
            save_current_figure(filename)
            plt.show()

if RUN_VISUALIZATIONS:
    plot_bar_charts(results_df)

# 19. Heatmaps

This cell creates heatmaps where rows are backbones and columns are heads.

It exists because heatmaps make it easy to see which backbone/head combinations are strong.

Expected output: saved heatmap figures.

In [ ]:
def plot_heatmaps(df: pd.DataFrame) -> None:
    plot_df = prepare_plot_df(df)
    for dataset_name in sorted(plot_df["dataset"].unique()):
        for k_shot in sorted(plot_df["k_shot"].unique()):
            subset = plot_df[(plot_df["dataset"] == dataset_name) & (plot_df["k_shot"] == k_shot)]
            if subset.empty:
                continue
            pivot = subset.pivot(index="backbone", columns="head", values="accuracy_percent")
            pivot = pivot.reindex(index=BACKBONE_NAMES, columns=HEAD_NAMES)
            plt.figure(figsize=(7, 5))
            image = plt.imshow(pivot.values, aspect="auto")
            plt.colorbar(image, label="Mean accuracy (%)")
            plt.xticks(range(len(pivot.columns)), pivot.columns)
            plt.yticks(range(len(pivot.index)), pivot.index)
            plt.title(f"Accuracy Heatmap - {dataset_name}, K={k_shot}")
            for i in range(pivot.shape[0]):
                for j in range(pivot.shape[1]):
                    value = pivot.values[i, j]
                    if not np.isnan(value):
                        plt.text(j, i, f"{value:.1f}", ha="center", va="center")
            filename = f"heatmap__{dataset_name}__K{k_shot}__N{NUM_EPISODES}.png"
            save_current_figure(filename)
            plt.show()

if RUN_VISUALIZATIONS:
    plot_heatmaps(results_df)

# 20. Runtime Analysis

According to the measured runtime table, the linear classifier is the slowest baseline head. This matches the expected behavior because it performs gradient-based optimization separately for every episode.

Ridge regression is the fastest baseline head in the measured table, while the prototypical classifier is between ridge and linear. This result shows that direct closed-form or non-parametric heads are much cheaper than repeatedly training a learned classifier.

Early stopping was included in the linear classifier to avoid unnecessary optimization after the support loss stopped improving. The timing table supports the general conclusion that the learned linear head remains the main runtime bottleneck even with this optimization.

The notebook reduces repeated work by caching extracted features, caching episodes, reusing each feature cache across all K values for the same dataset/backbone pair, skipping existing results, and using an efficient dual ridge implementation.

In [ ]:
def summarize_runtime(df: pd.DataFrame) -> pd.DataFrame:
    timing = df.groupby("head", as_index=False).agg(
        mean_seconds_per_episode=("mean_head_seconds_per_episode", "mean"),
        total_reported_block_seconds=("total_block_seconds", "sum"),
    ).sort_values("mean_seconds_per_episode", ascending=False)
    return timing

runtime_table = summarize_runtime(results_df)
runtime_table.to_csv(timing_path(), index=False)
display(runtime_table)

# 21. Training Dynamics of the Linear Classifier

This section plots the training curve for the learned Linear classifier.

The assignment asks to compare a low-shot case and a higher-shot case. This notebook records the first episode for K=1 and K=16 during benchmarking.

Expected output: training loss and accuracy curves if the benchmark has produced curve CSV files.

In [ ]:
def load_training_curve_files() -> List[Path]:
    return sorted(CURVE_DIR.glob("linear_curve__*.csv"))

def plot_training_curves() -> None:
    curve_files = load_training_curve_files()
    if not curve_files:
        print("No linear training curves were found.")
        print("Run the benchmark first. Curves are saved for the first episode of K=1 and K=16.")
        return
    for path in curve_files:
        curve_df = pd.read_csv(path)
        plt.figure(figsize=(7, 5))
        plt.plot(curve_df["epoch"], curve_df["loss"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Cross-entropy loss")
        plt.title(path.stem + " - Loss")
        plt.grid(True, alpha=0.3)
        save_current_figure(path.stem + "__loss.png")
        plt.show()
        plt.figure(figsize=(7, 5))
        plt.plot(curve_df["epoch"], 100.0 * curve_df["support_accuracy"], marker="o", label="Support")
        plt.plot(curve_df["epoch"], 100.0 * curve_df["query_accuracy"], marker="o", label="Query")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy (%)")
        plt.title(path.stem + " - Accuracy")
        plt.grid(True, alpha=0.3)
        plt.legend()
        save_current_figure(path.stem + "__accuracy.png")
        plt.show()

if RUN_VISUALIZATIONS:
    plot_training_curves()

## Training Dynamics Interpretation

The training curves compare the learned linear classifier in a 1-shot setting and a 16-shot setting.

In the 1-shot case, the classifier has only one labeled example per class. This is an extreme low-data regime. The support accuracy can increase quickly because the linear layer can fit the very small support set. However, high support accuracy does not necessarily imply high query accuracy. This gap indicates overfitting: the classifier learns the few available support examples, but it may not generalize well to unseen query samples.

In the 16-shot case, the classifier receives more examples per class. This usually produces a more stable optimization process and a smaller gap between support accuracy and query accuracy. The model has more information about the intra-class variation, so the learned decision boundaries are less dependent on individual samples.

Early stopping is useful in both cases. It reduces unnecessary optimization steps after the support loss stops improving. This is especially important because the linear classifier is trained again for every episode. In the full benchmark, avoiding unnecessary epochs has a direct effect on runtime.


# 22. t-SNE / UMAP Feature-Space Visualization

This section visualizes clusterability in the embedding space.

The goal is not to produce a quantitative score. The goal is to inspect whether images from the same class form visible clusters.

To keep this step light, it uses one sampled episode and only a limited number of classes when needed.

This cell defines helpers for t-SNE and UMAP visualizations.

Expected output: no output yet.

In [ ]:
def select_visualization_episode(dataset_name: str, k_shot: int) -> Dict[str, torch.Tensor]:
    episodes = load_episodes(dataset_name, k_shot, NUM_EPISODES)
    return episodes[0]

def limit_episode_classes(episode: Dict[str, torch.Tensor], max_classes: int) -> Dict[str, torch.Tensor]:
    class_ids = episode["class_ids"]
    keep_local = torch.arange(min(max_classes, len(class_ids)))
    support_mask = torch.isin(episode["support_labels"], keep_local)
    query_mask = torch.isin(episode["query_labels"], keep_local)
    return {
        "support_indices": episode["support_indices"][support_mask],
        "query_indices": episode["query_indices"][query_mask],
        "support_labels": episode["support_labels"][support_mask],
        "query_labels": episode["query_labels"][query_mask],
        "class_ids": episode["class_ids"][keep_local],
    }

def compute_2d_projection(features: np.ndarray, method: str = "tsne") -> np.ndarray:
    if method == "umap" and UMAP_AVAILABLE:
        reducer = umap.UMAP(n_components=2, random_state=SEED)
        return reducer.fit_transform(features)
    perplexity = min(30, max(5, features.shape[0] // 4))
    reducer = TSNE(n_components=2, random_state=SEED, init="pca", learning_rate="auto", perplexity=perplexity)
    return reducer.fit_transform(features)

def plot_feature_space(dataset_name: str, backbone_name: str, k_shot: int, max_classes: int = 10, method: str = "tsne") -> None:
    feature_pack = load_feature_cache(dataset_name, backbone_name, DEVICE)
    episode = select_visualization_episode(dataset_name, k_shot)
    episode = limit_episode_classes(episode, max_classes=max_classes)
    tensors = get_episode_tensors(feature_pack, episode, DEVICE)
    support_np = tensors["support_x_norm"].detach().cpu().numpy()
    query_np = tensors["query_x_norm"].detach().cpu().numpy()
    all_features = np.concatenate([support_np, query_np], axis=0)
    projection = compute_2d_projection(all_features, method=method)
    support_count = support_np.shape[0]
    support_2d = projection[:support_count]
    query_2d = projection[support_count:]
    support_y = tensors["support_y"].detach().cpu().numpy()
    query_y = tensors["query_y"].detach().cpu().numpy()
    plt.figure(figsize=(7, 6))
    plt.scatter(support_2d[:, 0], support_2d[:, 1], c=support_y, marker="o", label="Support", alpha=0.8)
    plt.scatter(query_2d[:, 0], query_2d[:, 1], c=query_y, marker="x", label="Query", alpha=0.8)
    plt.title(f"{method.upper()} - {dataset_name} / {backbone_name} / K={k_shot}")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.legend()
    plt.grid(True, alpha=0.3)
    filename = f"{method}__{dataset_name}__{backbone_name}__K{k_shot}.png"
    save_current_figure(filename)
    plt.show()
    del feature_pack
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

This cell generates t-SNE visualizations for selected configurations.

It exists to provide qualitative evidence about feature clusterability without adding a large computation burden.

Expected output: saved scatter plots.

In [ ]:
VISUALIZATION_BACKBONES = ["dinov2_small", "clip_rn50", "convnext_tiny"]
VISUALIZATION_K_SHOTS = [1, 2, 4, 16]
VISUALIZATION_MAX_CLASSES = 10
VISUALIZATION_METHOD = "tsne"

if RUN_VISUALIZATIONS:
    for dataset_name in DATASET_NAMES:
        for backbone_name in VISUALIZATION_BACKBONES:
            for k_shot in VISUALIZATION_K_SHOTS:
                plot_feature_space(dataset_name, backbone_name, k_shot, VISUALIZATION_MAX_CLASSES, VISUALIZATION_METHOD)
else:
    print("Visualizations are disabled.")

## Feature-Space Visualization Interpretation Guide

The t-SNE and UMAP visualizations provide a qualitative view of the learned feature space. Samples from the same class are expected to form compact clusters, while samples from different classes should be separated. Better cluster separation usually corresponds to better downstream classification accuracy. However, these visualizations should be interpreted qualitatively because they are two-dimensional projections of high-dimensional features.

# 23. Optional Task 5: Improved Few-Shot Head

This optional section implements a simple improved head called **Support-Centered Prototype**.

Intuition:

Regular Prototypical Classification compares query features to class means. The improved version first subtracts the global mean of the support set from both support and query features. This uses only the support set of the current episode. It does not use query labels and it does not reuse information across episodes.

This keeps the method fair for few-shot evaluation.

## Task 5 Method: Support-Centered Prototypical Classifier

For Task 5, I implemented an improved variant of the prototypical classifier. The method is called **Support-Centered Prototype**.

The standard prototypical classifier computes one prototype per class by averaging the support embeddings of that class. Then it classifies each query sample according to the nearest prototype.

My improvement adds a simple normalization step before computing the prototypes. For each episode, I first compute the mean feature vector of the whole support set. I subtract this support mean from both support features and query features. Then I L2-normalize the centered features and apply the usual prototypical classification rule.

This method is still a valid few-shot method because it uses only the support set of the current episode to compute the centering vector. It does not use query labels. It does not store information across episodes. It also keeps the backbone frozen. Therefore, it follows the local supervision and no-label-leakage requirements.

The intuition is that support centering removes an episode-specific global bias from the embedding space. This can make cosine similarity more focused on relative class structure rather than on the absolute location of the episode features. The method is also computationally cheap because it does not train extra parameters.

The Task 5 results show that the method is very fast. In the generated table, the average head time is around 0.0005 seconds per episode. Its accuracy increases consistently as K increases. For example, on Stanford Cars with CLIP-RN50, accuracy increases from about 24.83% at K=1 to about 59.96% at K=16. On EuroSAT with ConvNeXt-Tiny, it increases from about 53.38% at K=1 to about 79.86% at K=16.

Overall, this method is a simple proof of concept. It does not introduce a complex trainable adapter, but it improves the representation used by the prototypical classifier while preserving a fair few-shot evaluation protocol.


This cell implements the optional Task 5 head.

Expected output: no output yet.

In [ ]:
@torch.no_grad()
def predict_support_centered_prototype(support_x: torch.Tensor, support_y: torch.Tensor, query_x: torch.Tensor, num_classes: int) -> torch.Tensor:
    support_mean = support_x.mean(dim=0, keepdim=True)
    support_centered = support_x - support_mean
    query_centered = query_x - support_mean
    support_centered = F.normalize(support_centered, dim=1)
    query_centered = F.normalize(query_centered, dim=1)
    prototypes = compute_class_prototypes(support_centered, support_y, num_classes)
    scores = query_centered @ prototypes.T
    return torch.argmax(scores, dim=1)

This cell benchmarks the optional Task 5 method.

It uses the same pre-generated episodes as the main benchmark. The feature cache is loaded once per dataset/backbone pair and reused across all K values, while each K result is still saved separately for resume support.

Expected output: Task 5 result CSV files if `RUN_TASK5_BENCHMARK=True`.

In [ ]:
def run_task5_k_from_loaded_features(
    dataset_name: str,
    backbone_name: str,
    k_shot: int,
    num_episodes: int,
    feature_pack: Dict[str, torch.Tensor],
) -> pd.DataFrame:
    output_path = task5_result_path(dataset_name, backbone_name, k_shot, num_episodes)
    if output_path.exists():
        print("Task 5 result already exists, skipping:", output_path)
        return pd.read_csv(output_path)

    episodes = load_episodes(dataset_name, k_shot, num_episodes)
    scores = []
    times = []
    start_block = now_seconds()

    for episode in tqdm(episodes, desc=f"Task5 {dataset_name}/{backbone_name}/K={k_shot}"):
        tensors = get_episode_tensors(feature_pack, episode, DEVICE)
        start = now_seconds()
        predictions = predict_support_centered_prototype(
            tensors["support_x"],
            tensors["support_y"],
            tensors["query_x"],
            tensors["num_classes"],
        )
        times.append(elapsed_seconds(start))
        scores.append(compute_accuracy(predictions, tensors["query_y"]))

    total_seconds = elapsed_seconds(start_block)
    result_df = pd.DataFrame([{
        "dataset": dataset_name,
        "backbone": backbone_name,
        "head": "support_centered_prototype",
        "k_shot": k_shot,
        "num_episodes": num_episodes,
        "mean_accuracy": float(np.mean(scores)),
        "std_accuracy": float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0,
        "mean_head_seconds_per_episode": float(np.mean(times)),
        "total_block_seconds": float(total_seconds),
        "run_mode": RUN_MODE,
        "seed": SEED,
    }])

    result_df.to_csv(output_path, index=False)
    print("Saved Task 5 result:", output_path)
    return result_df

def run_task5_dataset_backbone_block(dataset_name: str, backbone_name: str, num_episodes: int) -> pd.DataFrame:
    print(f"Loading feature cache once for Task 5 {dataset_name}/{backbone_name}")
    feature_pack = load_feature_cache(dataset_name, backbone_name, DEVICE)
    frames = []

    for k_shot in K_SHOTS:
        frames.append(run_task5_k_from_loaded_features(
            dataset_name=dataset_name,
            backbone_name=backbone_name,
            k_shot=k_shot,
            num_episodes=num_episodes,
            feature_pack=feature_pack,
        ))

    del feature_pack
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.concat(frames, ignore_index=True)

def run_task5_block(dataset_name: str, backbone_name: str, k_shot: int, num_episodes: int) -> pd.DataFrame:
    print("Compatibility wrapper: loading features for a single Task 5 K block.")
    feature_pack = load_feature_cache(dataset_name, backbone_name, DEVICE)
    result_df = run_task5_k_from_loaded_features(
        dataset_name=dataset_name,
        backbone_name=backbone_name,
        k_shot=k_shot,
        num_episodes=num_episodes,
        feature_pack=feature_pack,
    )
    del feature_pack
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result_df

task5_frames = []

if RUN_TASK5_BENCHMARK:
    for dataset_name in DATASET_NAMES:
        for backbone_name in BACKBONE_NAMES:
            task5_frames.append(run_task5_dataset_backbone_block(
                dataset_name=dataset_name,
                backbone_name=backbone_name,
                num_episodes=NUM_EPISODES,
            ))

    task5_results = pd.concat(task5_frames, ignore_index=True)
    task5_combined_path = RESULTS_DIR / f"task5_results__{RUN_MODE}__N{NUM_EPISODES}.csv"
    task5_results.to_csv(task5_combined_path, index=False)
    display(task5_results)
else:
    print("Task 5 benchmark is disabled.")


# 24. Discussion

## Head Comparison

The three baseline heads have different strengths.

The prototypical classifier is simple and stable. It does not require gradient descent, so it is fast and has low risk of overfitting. However, it represents each class by only one mean vector, which can be restrictive.

Ridge regression is also efficient and benefits from regularization. It can model a more flexible linear mapping from features to class scores. In several settings, ridge can outperform the prototype head because it uses all support features jointly rather than relying only on class means.

The learned linear classifier is the most flexible baseline head. However, in very low-shot settings, especially K=1, it can overfit the support set. Its runtime is also higher because a new linear model is optimized for every episode. Reducing unnecessary per-epoch query evaluation and using early stopping makes this head more practical.

## Effect of K

Accuracy generally increases as K increases from 1 to 16. This is the expected behavior in few-shot learning. With more support examples per class, the classifier receives a better estimate of each class distribution.

The improvement is especially important for difficult settings such as Stanford Cars. This dataset is fine-grained: many car classes are visually similar, so one or two support examples may not capture enough variation. With K=16, the classifiers can form more reliable class representations.

## Feature-Space Visualization

The t-SNE plots provide a qualitative view of the embedding spaces. When samples from the same class form compact groups and different classes are separated, the downstream classifier usually performs better.

The plots should not be interpreted as formal metrics. t-SNE is sensitive to its parameters and only gives a two-dimensional projection of high-dimensional features. Still, the visualizations help explain why some backbone and dataset combinations are easier for the classifiers.

## Domain Shift

Stanford Cars and EuroSAT represent different types of difficulty.

Stanford Cars is a fine-grained natural image dataset. The challenge is that many classes are visually similar. Small differences between car models may be important.

EuroSAT is a satellite image dataset. The challenge is domain shift, because the backbones were mainly pre-trained on natural images rather than satellite imagery. Performance on EuroSAT therefore tests how well the learned visual features transfer to a different image domain.


# 25. Limitations

This benchmark has several limitations.

First, the results depend on the extracted feature representations. The backbones are frozen, so the experiment evaluates feature quality and classifier behavior, not end-to-end fine-tuning.

Second, the learned linear classifier depends on optimization hyperparameters such as learning rate, weight decay, maximum epochs, and early stopping. Different choices may change its performance and runtime.

Third, t-SNE visualizations are qualitative. They are useful for interpretation, but they should not replace the quantitative benchmark tables.

Fourth, the optional Task 5 method is intentionally simple. It is a proof of concept rather than a large trainable adaptation module. Its advantage is that it remains fair, fast, and fully local to each episode.

# 26. Conclusion
This notebook benchmarks frozen foundation-model features for few-shot classification on Stanford Cars and EuroSAT. The results show that both the backbone and the classifier head have a strong effect on performance.

Increasing K generally improves accuracy because each class is represented by more labeled examples. This effect is especially important in difficult fine-grained settings such as Stanford Cars.

Among the heads, prototype is simple and stable, ridge is a strong regularized baseline, and the learned linear classifier is more flexible but slower and more sensitive to overfitting. The Task 5 support-centered prototype method provides a lightweight extension that remains within the few-shot protocol and adds almost no computational cost.

Overall, the experiment demonstrates that strong frozen visual embeddings can support few-shot classification, but performance depends on feature separability, domain shift, and the choice of classification head.

All benchmark results were computed using cached feature embeddings rather than repeatedly forwarding raw images through the backbones. This significantly reduced the computational cost while preserving the evaluation protocol.

# 27. References

- Snell, J., Swersky, K., and Zemel, R. Prototypical Networks for Few-shot Learning. NeurIPS, 2017.
- Hoerl, A. E., and Kennard, R. W. Ridge Regression: Biased Estimation for Nonorthogonal Problems. Technometrics, 1970.
- Radford, A. et al. Learning Transferable Visual Models From Natural Language Supervision. ICML, 2021.
- Oquab, M. et al. DINOv2: Learning Robust Visual Features without Supervision. 2023.
- Liu, Z. et al. A ConvNet for the 2020s. CVPR, 2022.
- Van der Maaten, L. and Hinton, G. Visualizing Data using t-SNE. JMLR, 2008.
- McInnes, L., Healy, J., and Melville, J. UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction. 2018.